# stage5_non_vlm_hf_sweep

Clean Stage 5 no-VLM baseline run. Train-CV selects classifier settings; validation is evaluated once.

In [ ]:
from pathlib import Path
import subprocess, os, json, shutil
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_BRANCH = "next_research_vlm_benefit"
REPO_DIR = Path('/kaggle/working/vlm-for-insulator-defect-detection')
DATASET_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/kostyaryazanov/idid-coco-v3'),
    Path('/kaggle/input/idid-coco-v3'),
]
TRAIN_REL = Path('stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl')
VAL_REL = Path('stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl')
RUN_NAME = 'hf_sweep'
ARCHIVE = Path('/kaggle/working/stage5_non_vlm_hf_sweep.tar.gz')

def sh(args, cwd=None, check=True):
    print('$', ' '.join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f'Command failed {p.returncode}: {" ".join(str(a) for a in args)}')
    return p

def find_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    raise FileNotFoundError('stage3_regrouped_v2 train/val JSONL not found')


In [ ]:
gpu = sh(['nvidia-smi'], check=False)
if gpu.returncode != 0:
    raise RuntimeError('GPU is required for this run')
DATA_ROOT = find_data_root()
print('DATA_ROOT:', DATA_ROOT)
print('RUN_NAME:', RUN_NAME)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR])
sh(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR)
sh(['python', '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'accelerate', 'scikit-learn', 'pandas', 'numpy', 'pillow', 'timm', 'tabulate'], cwd=REPO_DIR)


In [ ]:
train_jsonl = DATA_ROOT / TRAIN_REL
val_jsonl = DATA_ROOT / VAL_REL
out_dir = f'reports/next_research/non_vlm_baselines/hf_sweep'
cache_dir = f'outputs/next_research/feature_cache/hf_sweep'
sh([
    'python', 'scripts/run_non_vlm_baselines.py',
    '--train-jsonl', train_jsonl,
    '--val-jsonl', val_jsonl,
    '--dataset-root', DATA_ROOT / 'stage3_regrouped_v2',
    '--out-dir', out_dir,
    '--cache-dir', cache_dir,
    '--hf-models=dinov2_base=facebook/dinov2-base:16,clip_b32=openai/clip-vit-base-patch32:32,clip_l14=openai/clip-vit-large-patch14:16,siglip_b16_224=google/siglip-base-patch16-224:16',
    '--timm-models=',
    '--classifiers', 'logreg,svm',
    '--continue-on-error',
    '--skip-timm-on-import-error',
], cwd=REPO_DIR)


In [ ]:
lb = REPO_DIR / out_dir / 'leaderboard_non_vlm.csv'
summary = REPO_DIR / out_dir / 'summary.md'
if lb.exists():
    df = pd.read_csv(lb)
    display(df)
if summary.exists():
    print(summary.read_text(encoding='utf-8')[:8000])


In [ ]:
if ARCHIVE.exists():
    ARCHIVE.unlink()
sh(['tar', '-czf', ARCHIVE, '-C', REPO_DIR, out_dir], check=True)
print('Archive:', ARCHIVE)
